# 01. Rotate

**Tier:** Deep. **Why you'd use this:** A 90° clockwise rotation looks like scattered index math. Transpose turns rows into columns; reversing each row finishes the turn without a second matrix.

**After this lesson you can:** rotate a square matrix in place by transpose then reverse; keep index math tied to the picture.
About 35 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. What does transpose do to rows and columns?
2. After transpose, what finishes a 90° clockwise rotation?

<details><summary>Answers</summary>

Transpose swaps `matrix[i][j]` with `matrix[j][i]`. Reversing each row then completes a 90° clockwise turn. Skim [Lists](../../course/04-collections/01-lists-lesson.ipynb) if nested indexing feels rusty.
</details>


## Recognize it

You must rearrange a square grid by a geometric rule: rotate 90°, reflect, or walk a fixed index pattern. The prompt says "rotate the image", "transpose", or "in-place" with a square matrix.

Signals: index mapping from old position to new, a square `n × n` grid, and often an in-place constraint. A look-alike that is not this pattern: flood-fill or island counting — those walk neighbors by value, not by a global geometric map. Spiral readout is a traversal order, not a rotation of the stored grid.


## Why it works this way

A 90° clockwise rotation sends the value at `(r, c)` to `(c, n - 1 - r)`. Writing that map with a second matrix is easy and uses `n²` extra cells. In place, you need a factorization into swaps you already trust.

Transpose swaps `(r, c)` with `(c, r)`. After transpose, the value that started at `(r, c)` sits at `(c, r)`. Reverse each row: position `(c, r)` moves to `(c, n - 1 - r)`. Compose the two maps and you get exactly `(r, c) → (c, n - 1 - r)`. That is the design reason for "transpose, then reverse rows" — each step is a familiar local rewrite, and together they match the rotation.

Transpose only walks `j > i` so each pair swaps once. Row reverse is linear per row. Total time is proportional to `n²`, which you must touch anyway to move every cell. Extra memory is a few temporaries for swaps. Building a new matrix is the same time with `n²` extra space; use that when the matrix is read-only or not square in a way in-place rotation forbids.

Off-by-one errors usually reverse columns instead of rows, or transpose with `j` starting at `0` and swap every pair twice back to the start. Draw a 3×3 before you code: label each cell, apply transpose on paper, then reverse each row, and confirm the picture matches a clockwise turn.

Counterclockwise 90° is the mirror recipe: transpose, then reverse each column (or reverse rows first, then transpose). Once you can see both factorizations on paper, the index formula stops feeling arbitrary.


## When to use it

Use transpose-plus-reverse for in-place 90° clockwise rotation of a square matrix. Reverse columns after transpose for counterclockwise. Use a second matrix when clarity matters or the grid is not square. Use a layer-by-layer cycle of four cells when you want to see the rotation as four-way swaps without naming transpose. Prefer spiral or diagonal walks when the task is to *read* an order, not to rewrite the grid.


## Brute force, and why it's slow

Allocate a new matrix and write each cell to its rotated coordinates. Count cell writes — they already match the in-place lower bound, but memory doubles.


In [1]:
def rotate_matrix_copy(matrix):
    writes = 0
    n = len(matrix)
    out = [[0] * n for _ in range(n)]
    for r in range(n):
        for c in range(n):
            out[c][n - 1 - r] = matrix[r][c]
            writes += 1
    return out, writes


for n in (3, 10, 30):
    grid = [[r * n + c for c in range(n)] for r in range(n)]
    _out, writes = rotate_matrix_copy(grid)
    print(n, writes)


3 9
10 100
30 900


## Key insight

Clockwise 90° equals transpose followed by reversing each row. Do both in place with pairwise swaps; no second matrix is required.


## Worked example

Rotate an `n × n` matrix 90° clockwise in place. The function returns `None` and mutates `matrix`.


In [2]:
def rotate_matrix(matrix: list[list[int]]) -> None:
    n = len(matrix)
    # 1. Transpose: swap across the diagonal.
    for i in range(n):
        for j in range(i + 1, n):
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    # 2. Reverse each row to finish the clockwise turn.
    for row in matrix:
        row.reverse()


grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
rotate_matrix(grid)
print(grid)


[[7, 4, 1], [8, 5, 2], [9, 6, 3]]


## Trace

`[[1, 2, 3], [4, 5, 6], [7, 8, 9]]`:

| step | matrix |
|---|---|
| start | `[[1, 2, 3], [4, 5, 6], [7, 8, 9]]` |
| after transpose | `[[1, 4, 7], [2, 5, 8], [3, 6, 9]]` |
| after reverse rows | `[[7, 4, 1], [8, 5, 2], [9, 6, 3]]` |


## Cost

Every cell moves once across transpose and once across row reverse, so time grows with `n²`. Extra memory is constant beyond the input matrix. The copy-based rotation uses the same time but allocates a full second grid.


## Variants

Practice walks diagonals in alternating directions. Related challenges (not solved here): [Spiral Matrix](../../challenges/problems/62-spiral-matrix.ipynb) reads a traversal order; [Rotate Image](../../challenges/problems/61-rotate-image.ipynb) is the challenge framing of this rotation.


## Common mistake

This bug is silent. The inner transpose loop starts `j` at `0` instead of `i + 1`, so every pair swaps twice and the matrix returns to the original layout before the row reverse. After "rotation" you get a horizontal flip of the input, not a clockwise turn.


In [3]:
def rotate_matrix_wrong(matrix):
    n = len(matrix)
    for i in range(n):
        for j in range(n):  # should be range(i + 1, n)
            matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]
    for row in matrix:
        row.reverse()
    return matrix


print(rotate_matrix_wrong([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))


[[3, 2, 1], [6, 5, 4], [9, 8, 7]]


The fix swaps only `j > i`. Transpose then becomes a real reflection across the diagonal, and reversing rows yields `[[7, 4, 1], [8, 5, 2], [9, 6, 3]]`.


In [4]:
grid = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
rotate_matrix(grid)
print(grid)


[[7, 4, 1], [8, 5, 2], [9, 6, 3]]


## Python idioms

`row.reverse()` mutates in place. `matrix[i][j], matrix[j][i] = matrix[j][i], matrix[i][j]` swaps without a temporary name. For a new-matrix version, a comprehension `[[matrix[r][c] for r in range(n - 1, -1, -1)] for c in range(n)]` builds the rotation in one expression when mutation is forbidden.


In [5]:
def rotate_matrix_new(matrix: list[list[int]]) -> list[list[int]]:
    n = len(matrix)
    return [[matrix[r][c] for r in range(n - 1, -1, -1)] for c in range(n)]


print(rotate_matrix_new([[1, 2, 3], [4, 5, 6], [7, 8, 9]]))


[[7, 4, 1], [8, 5, 2], [9, 6, 3]]


## Recap

- Clockwise 90° is transpose, then reverse each row.
- Transpose only over `j > i` or every pair undoes itself.
- Time is `n²`; in-place keeps extra memory constant.

Next: [practice](01-rotate-practice.ipynb).
